[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/05_Multi_Format_Export.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 高级多格式导出（Advanced Multi-Format Export）

## 概览

本高级 notebook 演示 Semantica 导出模块的全面能力，覆盖全部 **8 种导出格式**加报告生成。你将学会把同一张知识图谱同时导成多种格式、使用高级特性、并玩转方法注册表系统。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/export/)

### 先看主线

15 篇按目的地选出口，本篇全都要——同一张图、同一套嵌入、同一个本体，八种格式一个不落。十三步三段：

**全量导出（第 1–10 步）—— 八种格式逐个走**

备好样例数据（图 + 嵌入 + 本体）后，JSON → RDF → CSV → Graph → OWL → Vector → LPG → YAML 依次导出，再生成四种格式的报告。

**扩展与配置（第 11–12 步）—— 导出系统打开给你看**

方法注册表（注册自定义导出方法）和配置管理（环境变量/配置文件/程序化设置）。

**验收（第 13 步）**

逐个检查产物文件是否落地。（注：上游把第 13 步的说明误放成了代码单元，保持原样。）

### 你将学到什么

- 把知识图谱导出为全部 8 种支持格式
- 直接使用导出器类做精细控制
- 生成多格式专业报告
- 使用 RDF 序列化、校验和命名空间管理
- 注册并使用自定义导出方法
- 程序化配置导出设置
- 为向量库导出向量与嵌入
- 用 LPG 格式导出到图数据库

### 覆盖的导出格式

1. **JSON/JSON-LD** - 标准 JSON 和 JSON-LD 格式
2. **RDF** - Turtle、RDF/XML、JSON-LD、N-Triples、N3
3. **CSV** - 实体与关系的表格格式
4. **图格式** - GraphML、GEXF、DOT，面向可视化工具
5. **OWL** - 本体的 OWL/XML 与 Turtle
6. **向量** - 面向量库的 JSON、NumPy、Binary、FAISS
7. **LPG** - 面向图数据库的 Cypher 与 LPG
8. **YAML** - 语义网络与 schema 的 YAML
9. **报告** - HTML、Markdown、JSON、Text 报告

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

In [ ]:
!pip install semantica


In [ ]:
# Import core modules for building knowledge graph
from semantica.kg import GraphBuilder
from semantica.embeddings import EmbeddingGenerator
from semantica.ontology import OntologyGenerator
import os

# Create exports directory
os.makedirs("exports", exist_ok=True)


## 第 1 步：创建样例知识图谱与数据

创建带实体、关系、嵌入和本体的样例——全格式导出演示的总素材。

In [ ]:
builder = GraphBuilder()

entities = [
    {"id": "e1", "type": "Person", "name": "Alice", "properties": {"age": 30}},
    {"id": "e2", "type": "Person", "name": "Bob", "properties": {"age": 35}},
    {"id": "e3", "type": "Organization", "name": "Tech Corp", "properties": {"founded": 2010}},
]

relationships = [
    {"source": "e1", "target": "e2", "type": "knows"},
    {"source": "e1", "target": "e3", "type": "works_for"},
]

knowledge_graph = builder.build(entities + relationships)

embedding_generator = EmbeddingGenerator()
texts = [e["name"] for e in entities]
embeddings = embedding_generator.generate_embeddings(texts, data_type="text")

ontology_generator = OntologyGenerator()
ontology = ontology_generator.generate_from_graph(knowledge_graph)


## 第 2 步：导出为 JSON

类的写法和便捷函数两种姿势都用上。

**JSONExporter 特性：**
- 标准 JSON 序列化
- 带 @context 的 JSON-LD 格式支持
- 可配置缩进
- 元数据与溯源跟踪

In [ ]:
from semantica.export import JSONExporter

# Create JSON exporter with custom settings
json_exporter = JSONExporter(indent=2, include_metadata=True)

# Export to JSON format
json_exporter.export_knowledge_graph(knowledge_graph, "exports/output.json")

# Export to JSON-LD format
json_exporter.export_knowledge_graph(knowledge_graph, "exports/output.jsonld", format="json-ld")


## 第 3 步：导出为 RDF

一次导多种 RDF 格式（Turtle、RDF/XML、JSON-LD、N-Triples）——15 篇说过：一种数据、五种写法。

**RDFExporter 特性：**
- 多 RDF 格式支持
- 命名空间管理
- RDF 校验
- 格式互转

In [ ]:
from semantica.export import RDFExporter, RDFSerializer, RDFValidator

# Create RDF exporter
rdf_exporter = RDFExporter()

# Export to Turtle format (human-readable)
rdf_exporter.export_knowledge_graph(knowledge_graph, "exports/output.ttl", format="turtle")

# Export to RDF/XML format
rdf_exporter.export_knowledge_graph(knowledge_graph, "exports/output.rdf", format="rdfxml")

# Export to JSON-LD format
rdf_exporter.export_knowledge_graph(knowledge_graph, "exports/output.jsonld", format="jsonld")

# Export to N-Triples format
rdf_exporter.export_knowledge_graph(knowledge_graph, "exports/output.nt", format="ntriples")

# Using RDFSerializer for format conversion
serializer = RDFSerializer()
rdf_data = serializer.convert_kg_to_rdf(knowledge_graph)
turtle_string = serializer.serialize_to_turtle(rdf_data)

# Using RDFValidator for validation
validator = RDFValidator()
validation_result = validator.validate_rdf_syntax(turtle_string, format="turtle")
print(f"RDF validation: {validation_result.get('is_valid', False)}")


## 第 4 步：导出为 CSV

导成表格格式做分析。

**CSVExporter 特性：**
- 实体与关系分文件
- 可配置分隔符（逗号、制表符、分号）
- 自动生成表头
- 元数据序列化

In [ ]:
from semantica.export import CSVExporter

# Create CSV exporter with custom delimiter
csv_exporter = CSVExporter(delimiter=",")

# Export complete knowledge graph
csv_exporter.export_knowledge_graph(knowledge_graph, "exports/output")

# Export entities separately
entities = knowledge_graph.get("entities", [])
csv_exporter.export_entities(entities, "exports/entities.csv")

# Export relationships separately
relationships = knowledge_graph.get("relationships", [])
csv_exporter.export_relationships(relationships, "exports/relationships.csv")


## 第 5 步：导出为图格式（GraphML、GEXF、DOT）

面向可视化工具的导出——16/03 篇的可视化管线从这里拿料。

**GraphExporter 特性：**
- GraphML 格式（Cytoscape、yEd）
- GEXF 格式（Gephi）
- DOT 格式（Graphviz）
- 节点与边的属性映射

In [ ]:
from semantica.export import GraphExporter

# Create graph exporter
graph_exporter = GraphExporter()

# Export to GraphML format (for Cytoscape, yEd)
graph_exporter.export_knowledge_graph(knowledge_graph, "exports/output.graphml", format="graphml")

# Export to GEXF format (for Gephi)
graph_exporter.export_knowledge_graph(knowledge_graph, "exports/output.gexf", format="gexf")

# Export to DOT format (for Graphviz)
graph_exporter.export_knowledge_graph(knowledge_graph, "exports/output.dot", format="dot")


## 第 6 步：导出为 OWL

本体导出为 OWL。**注意：** `OWLExporter` 要的是本体结构，不是知识图谱——15 篇预告过的"楼与法"之别。

**OWLExporter 特性：**
- OWL/XML 格式
- Turtle 格式的 OWL
- 类层级导出
- 属性定义导出

In [ ]:
from semantica.export import OWLExporter

# Create OWL exporter with custom URI and version
owl_exporter = OWLExporter(ontology_uri="https://example.org/ontology/", version="1.0")

# Export complete ontology to OWL/XML
owl_exporter.export(ontology, "exports/output.owl", format="owl-xml")

# Export to OWL in Turtle format
owl_exporter.export(ontology, "exports/output_owl.ttl", format="turtle")

# Export only classes
classes = ontology.get("classes", [])
owl_exporter.export_classes(classes, "exports/classes.owl")

# Export only properties
properties = ontology.get("object_properties", [])
owl_exporter.export_properties(properties, "exports/properties.owl", property_type="object")


## 第 7 步：导出为向量格式

向量嵌入导成多种格式，喂给向量库。

**VectorExporter 特性：**
- JSON 格式
- NumPy 格式
- 二进制格式
- FAISS 格式
- 向量库集成（Weaviate、Qdrant）

In [ ]:
from semantica.export import VectorExporter

# Create vector exporter
vector_exporter = VectorExporter()

# Export to JSON format
vector_exporter.export(embeddings, "exports/output_vectors.json", format="json")

# Export to NumPy format
vector_exporter.export(embeddings, "exports/output_vectors.npy", format="numpy")

# Export to Binary format
vector_exporter.export(embeddings, "exports/output_vectors.bin", format="binary")

# Export to FAISS format
vector_exporter.export(embeddings, "exports/output_vectors.faiss", format="faiss")


## 第 8 步：导出为 LPG（带标签属性图）

导成 LPG 格式，进 Neo4j、Memgraph 这类图数据库——09 篇的专线在这里发车。

**LPGExporter 特性：**
- Cypher 查询格式
- 带标签属性图格式
- 节点/关系批量导出
- 索引生成

In [ ]:
from semantica.export import LPGExporter

# Create LPG exporter
lpg_exporter = LPGExporter()

# Export to Cypher format (for Neo4j, Memgraph)
lpg_exporter.export_knowledge_graph(knowledge_graph, "exports/output.cypher", format="cypher")

# Export to LPG format
lpg_exporter.export_knowledge_graph(knowledge_graph, "exports/output.lpg", format="lpg")


## 第 9 步：导出为 YAML

语义网络与 schema 导成 YAML。

**YAML 导出器特性：**
- 语义网络 YAML 导出
- schema YAML 导出
- 人类可读格式

In [ ]:
from semantica.export import SemanticNetworkYAMLExporter, YAMLSchemaExporter

# Using SemanticNetworkYAMLExporter for knowledge graphs
yaml_exporter = SemanticNetworkYAMLExporter()
yaml_exporter.export(knowledge_graph, "exports/output_network.yaml")

# Using YAMLSchemaExporter for ontology schemas
schema_exporter = YAMLSchemaExporter()
yaml_content = schema_exporter.export_ontology_schema(ontology)
with open("exports/output_schema.yaml", "w") as f:
    f.write(yaml_content)


## 第 10 步：生成报告

用 ReportGenerator 生成多格式专业报告——给人看的出口。

**ReportGenerator 特性：**
- 带样式的 HTML 报告
- Markdown 报告
- JSON 报告
- 纯文本报告
- 质量指标汇总

In [ ]:
from semantica.export import ReportGenerator

# Prepare report data
report_data = {
    "title": "Knowledge Graph Export Report",
    "summary": "Comprehensive export of knowledge graph to multiple formats",
    "knowledge_graph": {
        "entities": len(knowledge_graph.get("entities", [])),
        "relationships": len(knowledge_graph.get("relationships", []))
    },
    "formats_exported": ["JSON", "RDF", "CSV", "GraphML", "GEXF", "OWL", "Vector", "LPG", "YAML"],
    "export_timestamp": "2024-01-01T00:00:00Z"
}

# Create report generator
report_generator = ReportGenerator()

# Generate HTML report
report_generator.generate_report(report_data, "exports/report.html", format="html")

# Generate Markdown report
report_generator.generate_report(report_data, "exports/report.md", format="markdown")

# Generate JSON report
report_generator.generate_report(report_data, "exports/report.json", format="json")

# Generate Text report
report_generator.generate_report(report_data, "exports/report.txt", format="text")


## 第 11 步：方法注册表与自定义方法

导出系统的扩展点：用 MethodRegistry 注册并使用自定义导出方法。

**MethodRegistry 特性：**
- 注册自定义导出方法
- 列出可用方法
- 按名取方法
- 注销方法

In [ ]:
from semantica.export import MethodRegistry, method_registry, JSONExporter

# Define a custom export method
def custom_json_export(data, file_path, **kwargs):
    """Custom JSON export with additional formatting."""
    import json
    with open(file_path, 'w') as f:
        json.dump(data, f, indent=4, sort_keys=True)
    print(f"Custom export completed: {file_path}")

# Register custom method
MethodRegistry.register("json", "custom_formatted", custom_json_export)

# List all available methods
all_methods = method_registry.list_all()
print("Available methods:", all_methods)

# List methods for specific task
json_methods = method_registry.list_all("json")
print("JSON methods:", json_methods)

# Use registered method with JSONExporter
json_exporter = JSONExporter()
# The custom method can be used via the registry system
custom_method = method_registry.get("json", "custom_formatted")
if custom_method:
    custom_method(knowledge_graph, "exports/custom_output.json")


## 第 12 步：配置管理

用 ExportConfig 管理导出设置。

**ExportConfig 特性：**
- 环境变量支持
- 配置文件支持
- 程序化配置
- 按方法配置


### 回顾主线

八种格式加报告全部走完，扩展点（注册表）和配置也开过箱——15 篇的"按目的地选格式"是单选，本篇是全选加自定义：**导出系统从使用者视角进到了定制者视角**。

In [ ]:
from semantica.export import ExportConfig, export_config, JSONExporter

# Get current configuration
config = export_config.get("default")
print("Default config:", config)

# Set configuration programmatically
export_config.set("json", {"indent": 4, "include_metadata": True})
export_config.set("rdf", {"format": "turtle", "base_uri": "https://example.org/"})

# Get method-specific configuration
json_config = export_config.get_method_config("json")
print("JSON config:", json_config)

# Set method-specific configuration
export_config.set_method_config("csv", {"delimiter": "\t"})

# Use configured settings
json_exporter = JSONExporter(**export_config.get_method_config("json"))
json_exporter.export_knowledge_graph(knowledge_graph, "exports/output_configured.json")


In [ ]:
## Step 13: Verify All Exports

Verify that all exported files were created successfully.


In [ ]:
# List all exported files
export_files = [
    "exports/output.json",
    "exports/output.jsonld",
    "exports/output.ttl",
    "exports/output.rdf",
    "exports/output.nt",
    "exports/output.csv",
    "exports/entities.csv",
    "exports/relationships.csv",
    "exports/output.graphml",
    "exports/output.gexf",
    "exports/output.dot",
    "exports/output.owl",
    "exports/output_owl.ttl",
    "exports/output_vectors.json",
    "exports/output_vectors.npy",
    "exports/output.cypher",
    "exports/output_network.yaml",
    "exports/output_schema.yaml",
    "exports/report.html",
    "exports/report.md",
    "exports/report.json",
    "exports/report.txt"
]

print("📊 Export Summary:")
print("=" * 60)
for file in export_files:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f"✅ {file:50} ({size:>10,} bytes)")
    else:
        print(f"❌ {file:50} (not found)")

print("=" * 60)
print(f"Total files checked: {len(export_files)}")
print(f"Files created: {sum(1 for f in export_files if os.path.exists(f))}")
